AI as a Service (Kubernetes Variante)
---------------

Beispiel für einen **SGLang-Client**, der mit einem **SGLang Inference Server auf einem separaten Rechner** kommuniziert.

In diesem Szenario läuft **SGLang auf Kubernetes** als zentral betriebener Inference-Service. Bevor der Client Anfragen senden kann, muss jedoch zunächst das gewünschte **Modell als eigener Pod beziehungsweise Container gestartet** werden. Erst danach steht es über eine **OpenAI-kompatible API** für die Clients zur Verfügung.

Im Unterschied zu Ollama unterstützt **SGLang pro Container in der Regel nur ein einzelnes LLM**. Sollen mehrere Modelle parallel betrieben werden, werden diese deshalb üblicherweise als **separate Deployments oder Pods** gestartet und über unterschiedliche Services, Endpoints oder Ports bereitgestellt.

Folgende Befehle in der `rPodman Sandbox` ausführen:

    kubectl create ns qwen-instruct
    kubectl apply -n qwen-instruct -f https://github.com/mc-b/lernvirt/raw/refs/heads/main/examples/aiaas/k8s/qwen-instruct-deployment.yaml
    kubectl apply -n qwen-instruct -f https://github.com/mc-b/lernvirt/raw/refs/heads/main/examples/aiaas/k8s/qwen-instruct-service.yaml

Und um den richtigen Port zu finden

    kubectl -n qwen-instruct get services


- - -

Der folgende Code zeigt, wie aus einem **Python-Jupyter-Notebook** über diese OpenAI-kompatible API auf einen betriebenen SGLang-Service zugegriffen wird.

Die Verbindung erfolgt über den API-Endpoint des Servers. Der verwendete API-Key dient dabei lediglich als Platzhalter.

Die Funktion `ask` kapselt einen einfachen Chat-Request. Da ein SGLang-Container typischerweise genau ein Modell bereitstellt, dient der Modellname hier in erster Linie der expliziten Adressierung des geladenen Dienstes beziehungsweise der Kompatibilität zur OpenAI-Schnittstelle. Der übrige Code muss dadurch nicht angepasst werden.

In [ ]:
from openai import OpenAI
import time

def ask(model, prompt, port=31868, max_tokens=2048):

    client = OpenAI(
        base_url=f"http://10.3.24.17:{port}/v1",
        api_key="sglang"
    )

    start = time.perf_counter()

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=max_tokens,
    )

    end = time.perf_counter()
    duration = end - start

    answer = response.choices[0].message.content

    prompt_tokens = response.usage.prompt_tokens
    completion_tokens = response.usage.completion_tokens
    total_tokens = response.usage.total_tokens

    tokens_per_sec = completion_tokens / duration

    print(f"Antwortzeit: {duration:.2f} Sekunden")
    print(f"Prompt Tokens: {prompt_tokens}")
    print(f"Completion Tokens: {completion_tokens}")
    print(f"Total Tokens: {total_tokens}")
    print(f"Decode Speed: {tokens_per_sec:.2f} tokens/s")

    return answer


Dieses Codebeispiel sendet eine Anfrage an das Sprachmodell `Qwen/Qwen2.5-0.5B-Instruct`, das auf einem **SGLang-Inference-Server** betrieben wird. Das kompakte Instruct-Modell eignet sich für einfache Dialoge, kurze Erklärungen und grundlegende technische Fragestellungen. Der Zugriff erfolgt über die **OpenAI-kompatible API** von SGLang.

Bei der Interpretation der gemessenen Antwortzeit ist zu beachten, dass der **erste Request nach dem Start eines Containers** oft langsamer ist. In dieser Phase werden Modellgewichte geladen und die Inferenz-Engine initialisiert. Dieser Vorgang wird als **Kaltstart** bezeichnet. Nachfolgende Anfragen sind meist schneller, da das Modell bereits im Speicher liegt (**Warm-Start**).

Im Unterschied zu Ollama betreibt **SGLang typischerweise nur ein Modell pro Container**. Mehrere Modelle werden daher meist über mehrere Container oder Instanzen betrieben, die über unterschiedliche Ports angesprochen werden.

Für Performance-Analysen lohnt es sich zusätzlich, **auf dem Server die Container-Logs zu betrachten**. Dort zeigt SGLang unter anderem den **Token-Durchsatz (Tokens pro Sekunde)** sowie weitere Laufzeitmetriken der Inferenz an.


In [ ]:
print(ask(
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Erkläre HTTP Codes kurz."
))

- - -
`HuggingFaceTB/SmolLM2-1.7B-Instruct` ist ein kompaktes Instruct-Sprachmodell aus der SmolLM2-Familie von Hugging Face. Mit rund 1.7 Milliarden Parametern gehört es zu den kleineren LLMs und ist darauf ausgelegt, auf moderater Hardware effizient zu laufen, etwa auf einer einzelnen GPU oder leistungsfähigen CPU-Systemen.

Das Modell wurde speziell für **dialogorientierte Aufgaben und instruktionbasierte Prompts** trainiert. Es eignet sich für kurze Erklärungen, einfache Programmierhilfe, Zusammenfassungen oder strukturierte Antworten auf technische Fragen. Durch seine geringe Modellgrösse reagiert es meist schnell und verursacht deutlich geringere Ressourcen- und Latenzanforderungen als grössere Modelle.

SmolLM2-Modelle werden häufig in **lokalen Inferenz-Setups, Edge-Umgebungen oder experimentellen LLM-Workflows** eingesetzt, bei denen ein guter Kompromiss zwischen Modellqualität, Geschwindigkeit und Hardwarebedarf wichtig ist.

In [ ]:
print(ask(
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "Schreibe ein kurzes Python Beispiel für einen REST Client.",
    port=31838
))

- - -

### Kubernetes Zugriff 

Mit einem kleinen Trick kann Jupyter Lab direkt auf Kubernetes auf der DGX Spark zugreifen.

1. Datei config.txt anlegen
2. In der `rPodman Sandbox` - `cat /etc/rancher/k3s/k3s.yaml`
3. Inhalt in config.txt ablegen und `server:` Zeile auf WireGuard IP-Adresse ändern


In [ ]:
%%bash
kubectl --kubeconfig config.txt get all

Log Ausgabe

In [ ]:
%%bash
kubectl --kubeconfig config.txt logs deployment/qwen-sglang-model

In [ ]:
%%bash
kubectl --kubeconfig config.txt logs deployment/smollm2-sglang-model

--- 

## Weitere Modelle

`Qwen/Qwen2.5-Coder-0.5B` ist ein sehr kompaktes Code-Sprachmodell der Qwen-2.5-Familie. Mit rund 0.5 Milliarden Parametern ist es auf schnelle Inferenz und geringen Ressourcenbedarf ausgelegt und eignet sich für einfache Programmieraufgaben, kurze Codebeispiele oder grundlegende Erklärungen zu Softwarekonzepten.

Aufgrund seiner kleinen Modellgrösse kann es jedoch gelegentlich zu instabiler Textgenerierung kommen, etwa zu Wiederholungen einzelner Tokens oder semantischem Drift. In der Praxis empfiehlt es sich daher, die Generierung über Parameter wie `max_tokens`, `temperature` oder `top_p` (z.B. 0.9) zu begrenzen, um stabilere und besser kontrollierbare Antworten zu erhalten.


In [ ]:
print(ask(
    "Qwen/Qwen2.5-Coder-0.5B",
    "Schreibe ein kurzes Python Beispiel für einen REST Client.",
    port=30934
))

Mit `max_tokens=64` wird die maximale Länge der Modellantwort begrenzt. Das verhindert unnötig lange Ausgaben und reduziert gleichzeitig die Wahrscheinlichkeit von Token-Wiederholungen, die bei sehr kleinen Modellen auftreten können.

In [ ]:
print(ask(
    "Qwen/Qwen2.5-Coder-0.5B",
    """What does this Python code do?
for i in range(3):
    print(i)
""",
    port=30934, max_tokens=64
))

In [ ]:
print(ask(
    "HuggingFaceTB/SmolLM2-135M-Instruct",
    "Schreibe ein kurzes Python Beispiel für einen REST Client.",
    port=31697,
    max_tokens=512
))

## Apertus

Apertus ist ein Sprachmodell mit 70 Milliarden bzw. 8 Milliarden Parametern, das die Grenzen vollständig offener, mehrsprachiger und transparenter Modelle erweitern soll. Das Modell unterstützt über 1000 Sprachen und lange Kontexte, verwendet ausschließlich vollständig konforme und offene Trainingsdaten und erzielt eine vergleichbare Leistung wie Modelle, die intern trainiert wurden

* [Hugging Face](https://huggingface.co/swiss-ai/Apertus-8B-Instruct-2509)

In [ ]:
print(ask(
    "swiss-ai/Apertus-8B-Instruct-2509",
    "Schreibe ein kurzes Python Beispiel für einen REST Client.",
    port=31867,
    max_tokens=512
))

- - -

### Halluzination in LLMs

![](halluzination.png)

---

Dieses Beispiel zeigt eine typische **Halluzination eines LLMs**: Das Modell erzeugt eine scheinbar plausible Antwort über John F. Kennedy, erfindet jedoch mehrere Fakten (z. B. falsche Lebensdaten und erfundene Begriffe) und präsentiert diese selbstbewusst als korrekt.

In [ ]:
print(ask(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "Wer war John F. Kennedy in Deutschland",
    port=31017,
    max_tokens=512
))

- - -

### Low Level Zugriff mittels curl

Der gezeigte **Low-Level API-Aufruf** sendet einen Prompt direkt an einen Text-Generator-Endpoint und gibt neben dem generierten Text auch interne Metadaten der Inferenz zurück. Die JSON-Antwort enthält mehrere Felder, die unterschiedliche Aspekte der Modellantwort beschreiben.

**text**
Dieses Feld enthält den **vom Modell generierten Klartext**. Es ist die dekodierte, für Menschen lesbare Antwort auf den Prompt *“What does NVIDIA love?”*. Der Text entsteht durch probabilistische Token-Sampling-Verfahren gemäss den angegebenen Sampling-Parametern (hier z. B. `temperature: 0.7`).

**output_ids**
Dies ist die **Tokenisierte Darstellung der generierten Antwort**. Jeder Integer repräsentiert ein Token im verwendeten Tokenizer-Vokabular des Modells. Die Sequenz entspricht exakt der Tokenfolge, aus der das Modell den Text dekodiert hat. Diese Information wird typischerweise für Debugging, Re-Ranking oder Reproduzierbarkeit der Generierung verwendet.

**meta_info**
Dieses Objekt enthält **technische Metadaten der Inferenz**.

* **id** – eindeutige Request- bzw. Generation-ID zur Nachverfolgung.
* **finish_reason** – Grund, warum die Generierung gestoppt wurde.

  * `type: "length"` bedeutet, dass die Ausgabe das gesetzte Limit erreicht hat.
  * `length: 100` zeigt, dass `max_new_tokens` ausgeschöpft wurde.
* **prompt_tokens** – Anzahl Tokens des eingegebenen Prompts.
* **completion_tokens** – Anzahl Tokens, die das Modell generiert hat.
* **cached_tokens** – Tokens aus dem Prompt, die aus einem KV-Cache wiederverwendet wurden (optimiert die Inferenz bei wiederholten Prompts).
* **cached_tokens_details** – zusätzliche Cache-Informationen (hier nicht gesetzt).
* **weight_version** – Version der verwendeten Modellgewichte.
* **total_retractions** – Anzahl interner Token-Rücknahmen (bei bestimmten Decoding-Strategien möglich).
* **e2e_latency** – gesamte Latenz der Anfrage in Sekunden (End-to-End).
* **response_sent_to_client_ts** – Unix-Timestamp, wann die Antwort an den Client gesendet wurde.

Zusammengefasst zeigt die Ausgabe **(1) den generierten Text**, **(2) die zugrunde liegenden Token-IDs** und **(3) interne Inferenz- und Performance-Metadaten**, die typischerweise bei Low-Level-LLM-Endpoints für Debugging, Monitoring oder Performance-Analyse bereitgestellt werden.

In [ ]:
%%bash
curl -sS -X POST http://10.3.24.17:31868/generate \
      -H "Content-Type: application/json" \
      -d '{
          "text": "What does NVIDIA love?",
          "sampling_params": {
              "temperature": 0.7,
              "max_new_tokens": 100
          }
      }' | jq

- - - 

### Lasttest mit hey

Falls `hey` noch Installiert wurde:

        wget -nv "https://hey-release.s3.us-east-2.amazonaws.com/hey_linux_amd64" -O hey
        chmod 755 hey
        sudo mv hey /usr/local/bin/


Der ursprüngliche `curl`-Befehl sendet eine einzelne POST-Anfrage an den Endpoint `/generate`. Mit **hey** kann derselbe Request vollständig nachgebildet und gleichzeitig für Lasttests verwendet werden.

Bedeutung der Parameter:
* **-n 10** – Gesamtanzahl der Requests, die gesendet werden
* **-c 5** – Anzahl paralleler Clients / Verbindungen

Im Unterschied zu `curl` gibt `hey` nach Abschluss eine **Performance-Zusammenfassung** aus, typischerweise:
* durchschnittliche und maximale **Latenz**
* **Requests pro Sekunde (throughput)**
* **Latency-Perzentile** (p50, p90, p99)
* Verteilung der Antwortzeiten
* Fehlerquote

Damit lässt sich derselbe Inferenz-Endpoint nicht nur funktional testen, sondern auch hinsichtlich **Skalierbarkeit und Antwortzeit unter Parallel-Last** evaluieren.

In [ ]:
%%bash
hey -n 10 -c 5 -m POST \
  -H "Content-Type: application/json" \
  -d '{
        "text": "What does NVIDIA love?",
        "sampling_params": {
          "temperature": 0.7,
          "max_new_tokens": 100
        }
      }' \
  http://10.3.24.17:31868/generate